In [0]:
%pip install sqlglot
%restart_python

In [0]:
%sql
CREATE OR REPLACE TABLE  dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_dataentry_definition 
as (
    WITH API AS (
    SELECT
        DataItem_unique_Id,
        universe_id,
        DataItem_id,
        DataItem_name,
        case when endswith(universe_name, '.unx') then left(universe_name, length(universe_name) - 4) else universe_name end as universe_name,
        DataItem_type,
        DataItem_description,
        DataItem_dataType,
        -- Process DataItem_path: split by '\', remove everything after '|' in each part, then join back with '\'
        concat(
        '\\',
        array_join(
            transform(
            slice(
                split(DataItem_path, '\\\\'),
                1,
                size(split(DataItem_path, '\\\\')) - 1
            ),
            part -> split(part, '\\|')[0]
            ),
            '\\'
        )
        ) AS DataItem_path,
        universe_path,
        universe_type
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.universe_datafields_api_list
    ),
    idt_definition AS (
    SELECT
        universe_name,
        record_type AS DataItem_type,
        Cuid AS DataItem_id,
        Name AS DataItem_name,
        Data_Type AS DataItem_dataType,
        Description AS DataItem_description,
        `Select` AS sql_definition,
        Path AS DataItem_path
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.universe_unvx_definitions
    )
    select API.*, idt_definition.sql_definition, idt_definition.DataItem_name as DataItem_name_definition, left(idt_definition.DataItem_path, length(idt_definition.DataItem_path) - 1) as DataItem_path_definition 
    from API 
    left join idt_definition 
    on upper(api.universe_name) = upper(idt_definition.universe_name)
    and (
        API.DataItem_id = idt_definition.DataItem_id 
        or (
        upper(API.DataItem_name) = upper(idt_definition.DataItem_name) 
        and upper(API.DataItem_path) = upper(left(idt_definition.DataItem_path, length(idt_definition.DataItem_path) - 1))
        )
    )
)

In [0]:
%sql
SELECT
  universe_name,
  CASE WHEN sql_definition IS NOT NULL THEN 'Found' ELSE 'Null' END AS sql_definition_status,
  COUNT(*) AS count
FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_dataentry_definition
GROUP BY universe_name, sql_definition_status
union ALL
select universe_name, 'apt_total' as sql_definition_status, count(distinct DataItem_id) as count
FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.universe_datafields_api_list
group by universe_name 
ORDER BY universe_name, sql_definition_status


Usage Note: 
Universe ID matching is not reliable, better to use Universe Nameremove **.unx & _old_ud**t
-     AM Tool Universe
-     AM Tool Universe.unx
-     AM Tool Universe_old_udt

**_Data Item matching need to combine path_**

In [0]:

import re
from pyspark.sql.functions import udf, expr
from pyspark.sql.types import ArrayType, StringType

df = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_dataentry_definition")

# Extract all TABLE.COLUMN or SCHEMA.TABLE.COLUMN references
# Strips away functions like SUM(), COUNT(), DECODE(), NVL(), CASE, TO_DATE(), @Prompt() etc.
def extract_table_refs(sql_def):
    if not sql_def:
        return None
    # Match dotted identifiers: 2+ parts separated by dots
    pattern = r'[A-Za-z_][A-Za-z0-9_#$]*\.[A-Za-z_][A-Za-z0-9_#$]*(?:\.[A-Za-z_][A-Za-z0-9_#$]*)*'
    matches = re.findall(pattern, sql_def)
    # Deduplicate while preserving order
    seen = set()
    result = []
    for m in matches:
        if m not in seen:
            seen.add(m)
            result.append(m)
    return result if result else None

extract_table_refs_udf = udf(extract_table_refs, ArrayType(StringType()))

# Extract table name from each ref by taking everything before the last dot
def get_table_names(refs):
    if not refs:
        return None
    tables = []
    seen = set()
    for ref in refs:
        table = ref.rsplit('.', 1)[0]  # everything before last dot
        if table and table not in seen:
            seen.add(table)
            tables.append(table)
    return tables if tables else None

extract_table_refs_udf = udf(extract_table_refs, ArrayType(StringType()))
get_table_names_udf = udf(get_table_names, ArrayType(StringType()))

df_with_refs = df.withColumn("table_column_refs", extract_table_refs_udf(df["sql_definition"])) \
    .withColumn("SQL_Tables", get_table_names_udf("table_column_refs")) \
    .withColumn("SQL_Tables_str", expr("array_join(SQL_Tables, ' | ')"))

# Keep all original columns + SQL_Tables_str, write back to the table
df_final = df_with_refs.select(*df.columns, "SQL_Tables_str")
df_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_dataentry_definition")
